# Práctica 7 — SVM y K-NN en Datos con Geometría Variable
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---
**Objetivo:** Comparar SVM y K-NN en datasets sintéticos con diferente geometría (blobs, moons, circles). Analizar cuándo cada algoritmo falla y cuándo sobresale.

**Datasets:** make_blobs · make_moons · make_circles (scikit-learn)

⚠️ **Instrucciones:**
- Celdas marcadas con `# 🔧 TU CÓDIGO` debes completarlas.
- Responde las preguntas ❓ en celdas Markdown.
- Guarda una copia en Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets        import make_blobs, make_moons, make_circles
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics         import accuracy_score
from sklearn.pipeline        import Pipeline

print('✅ Librerías cargadas')

## Parte 1 — Generación y Visualización de Datasets

In [ ]:
np.random.seed(42)
datasets = {
    'Blobs (lineal)':        make_blobs(n_samples=300, centers=2, cluster_std=1.5, random_state=42),
    'Moons (curvo)':         make_moons(n_samples=300, noise=0.15, random_state=42),
    'Circles (concéntrico)': make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=42),
}
print('✅ Datasets generados')
for nombre, (X_d, y_d) in datasets.items():
    print(f'  {nombre}: X={X_d.shape}, clases={np.unique(y_d)}')

In [ ]:
# 🔧 TU CÓDIGO
# Visualiza los 3 datasets en 1 fila de 3 subplots
# Colorea clase 0 = rojo, clase 1 = azul

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nombre, (X_d, y_d)) in zip(axes, datasets.items()):
    # ___ tu código aquí ___
    pass

plt.suptitle('Tres geometrías de clasificación — Datos sintéticos', fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Predicciones antes de modelar
*(Responde aquí en Markdown)*

1. ¿Qué geometría resolverá bien SVM lineal? ¿Cuál fallará?
2. ¿Qué geometría resolverá bien K-NN (k pequeño)?

## Parte 2 — SVM vs K-NN en cada Geometría

In [ ]:
# 🔧 TU CÓDIGO
# Para cada dataset:
#   1. Divide train/test (70/30, random_state=42)
#   2. Escala con StandardScaler
#   3. Entrena SVM lineal, SVM RBF y K-NN (k=5)
#   4. Visualiza la frontera de decisión y muestra accuracy en el título

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
modelos = {
    'SVM Lineal': SVC(kernel='linear', C=1),
    'SVM RBF':    SVC(kernel='rbf', C=1, gamma='scale'),
    'K-NN k=5':  KNeighborsClassifier(n_neighbors=5),
}

for row, (nombre_data, (X_d, y_d)) in enumerate(datasets.items()):
    X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.30, random_state=42)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    for col, (nombre_mod, clf) in enumerate(modelos.items()):
        ax = axes[row][col]
        # ___ entrenar clf ___
        # ___ visualizar frontera de decisión ___
        # ___ añadir título con acc ___
        pass

plt.suptitle('SVM Lineal vs SVM RBF vs K-NN — Tres geometrías', fontsize=11)
plt.tight_layout()
plt.show()

### ❓ Análisis de resultados
*(Responde aquí en Markdown)*

1. ¿SVM lineal puede resolver Moons y Circles? ¿Por qué sí o no?
2. ¿SVM RBF maneja los 3 datasets?
3. ¿K-NN (k=5) resuelve los 3 o hay alguna geometría donde falla?

## Parte 3 — GridSearchCV en el Dataset Circles

In [ ]:
# 🔧 TU CÓDIGO
# Dataset: Circles (el más difícil para kernel lineal)
X_c, y_c = datasets['Circles (concéntrico)']
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.30, random_state=42)

# Pipeline SVM
pipe_svm = Pipeline([('scaler', StandardScaler()), ('svm', SVC())])
param_svm = {
    'svm__kernel': ['rbf', 'poly'],
    'svm__C':      [0.1, 1, 10],
    'svm__gamma':  ['scale', 0.1, 1],
}

# Pipeline K-NN
pipe_knn = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])
param_knn = {
    'knn__n_neighbors': [3, 5, 7, 11],
    'knn__weights':     ['uniform', 'distance'],
}

grid_svm = ___________
grid_knn = ___________
grid_svm.fit(X_tr, y_tr)
grid_knn.fit(X_tr, y_tr)

print('── GridSearchCV — Dataset Circles ────────────────────────')
print(f'SVM | Params: {grid_svm.best_params_} | CV-5: {grid_svm.best_score_:.4f} | Test: {grid_svm.score(X_te, y_te):.4f}')
print(f'KNN | Params: {grid_knn.best_params_} | CV-5: {grid_knn.best_score_:.4f} | Test: {grid_knn.score(X_te, y_te):.4f}')

### ❓ Preguntas Parte 3
*(Responde aquí en Markdown)*

1. ¿Qué kernel SVM ganó en Circles? ¿Con qué valores de C y gamma?
2. ¿Cuál modelo (SVM o K-NN) tuvo mejor accuracy en Circles?

## Parte 4 — Efecto del Ruido

In [ ]:
# 🔧 TU CÓDIGO
# Para make_moons con noise = [0.05, 0.15, 0.25, 0.40]:
#   - Entrena SVM RBF y K-NN (k=5) en cada versión
#   - Calcula CV-5 accuracy
# Grafica accuracy vs nivel de ruido para ambos modelos

niveles_ruido = [0.05, 0.15, 0.25, 0.40]
acc_svm_ruido = []
acc_knn_ruido = []

for noise in niveles_ruido:
    X_n, y_n = make_moons(n_samples=300, noise=noise, random_state=42)
    # ___ calcular CV-5 para SVM RBF y K-NN k=5 con StandardScaler ___
    pass

plt.figure(figsize=(8, 4))
plt.plot(niveles_ruido, acc_svm_ruido, 'o-', color='#3b82f6', lw=2, label='SVM (RBF, C=1)')
plt.plot(niveles_ruido, acc_knn_ruido, 's-', color='#10b981', lw=2, label='K-NN (k=5)')
plt.xlabel('Nivel de ruido (noise)')
plt.ylabel('CV-5 Accuracy')
plt.title('Efecto del ruido — SVM vs K-NN en make_moons')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ❓ Análisis del ruido
*(Responde aquí en Markdown)*

1. ¿Cuál modelo es más robusto al ruido?
2. ¿Por qué SVM es teóricamente más robusto que K-NN al ruido?

## Parte 5 — Tabla Comparativa Final

In [ ]:
# 🔧 TU CÓDIGO
# Completa con los resultados de las partes anteriores

import pandas as pd

tabla = pd.DataFrame({
    'Dataset':        ['Blobs (lineal)', 'Moons (curvo)', 'Circles (concéntrico)'],
    'SVM Lineal acc': [___, ___, ___],
    'SVM RBF acc':    [___, ___, ___],
    'K-NN (k=5) acc': [___, ___, ___],
})
print('── Tabla Comparativa — Accuracy ──────────────────────────')
print(tabla.to_string(index=False))

## 📝 Conclusión Integradora — ¿Cuándo elegir SVM o K-NN?

*(Escribe aquí tu párrafo de conclusiones)*

Basándote en los resultados de esta práctica y las semanas 5 y 6, responde:
- ¿En qué tipo de problema elegirías SVM? ¿Y K-NN?
- Menciona al menos 3 criterios concretos: geometría de datos, ruido, escalabilidad, interpretabilidad.